In [1]:
import os, sys, warnings

import earthaccess
from osgeo import gdal

import pandas as pd
import numpy as np
import xarray as xr
import math
import glob

import rasterio as rio

import netCDF4 as nc
from datetime import datetime, timedelta, timezone

from scipy import ndimage as ndi
from scipy.ndimage import binary_fill_holes, center_of_mass, distance_transform_edt
from scipy.interpolate import PchipInterpolator  # monotone, shape-preserving

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from scipy import linalg
from scipy.signal import savgol_filter
import scipy.ndimage as ndi
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation, rotate, shift
from skimage.transform import hough_line, hough_line_peaks
from skimage.restoration import (
    denoise_tv_chambolle,
    denoise_bilateral,
    denoise_wavelet,
    estimate_sigma,
    inpaint_biharmonic
)

# This will ignore some warnings caused by holoviews
warnings.simplefilter('ignore') 

sys.path.append('../../EMIT-Data-Resources/python/modules/')
from emit_tools import emit_xarray, ortho_xr

sys.path.append('../')
from config import *
from REFERENCE_PLANTS import REFERENCE_PLANTS

sys.path.append('../../datasets/')
import get_geosfp
import get_campd
import amf_compute
import importlib


# Reload the module
importlib.reload(get_campd)
importlib.reload(get_geosfp)
importlib.reload(amf_compute)

get_emission_rate = get_campd.get_emission_rate
get_emissions = get_campd.get_emissions

get_geosfp_wind = get_geosfp.get_geosfp_wind
get_geosfp_tph = get_geosfp.get_geosfp_tph
get_geosfp_wind_agl = get_geosfp.get_geosfp_wind_agl

PREFIXES = {'OBS': 'EMIT_L1B_OBS_001_',
 'L1B': 'EMIT_L1B_RAD_001_',
 'MASK': 'EMIT_L2A_MASK_001_',
 'L2A': 'EMIT_L2A_RFL_001_'}

help(emit_xarray)

Help on function emit_xarray in module emit_tools:

emit_xarray(filepath, ortho=False, qmask=None, unpacked_bmask=None)
    This function utilizes other functions in this module to streamline opening an EMIT dataset as an xarray.Dataset.
    
    Parameters:
    filepath: a filepath to an EMIT netCDF file
    ortho: True or False, whether to orthorectify the dataset or leave in crosstrack/downtrack coordinates.
    qmask: a numpy array output from the quality_mask function used to mask pixels based on quality flags selected in that function. Any non-orthorectified array with the proper crosstrack and downtrack dimensions can also be used.
    unpacked_bmask: a numpy array from  the band_mask function that can be used to mask band-specific pixels that have been interpolated.
    
    Returns:
    out_xr: an xarray.Dataset constructed based on the parameters provided.



In [2]:
!ls /orcd/data/dvaron/001/kgauld/GEOS_FP/*3d*

/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230330_1030.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230403_0730.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230620_1030.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230820_1030.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230824_0430.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230824_0730.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20231003_0730.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20240216_1030.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20250424_0730.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20250428_0430.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20250613_1030.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.f

In [3]:
!ls /orcd/data/dvaron/001/kgauld/GEOS_FP/*20230824*

/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg1_2d_slv_Nx.20230824_0830.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg1_2d_slv_Nx.20230824_1730.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230824_0430.V01.nc4
/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230824_0730.V01.nc4


In [55]:
loc_name = 'RIYADH_PLANT_9'
loc_data = LOCS['RIYADH_PLANT_9']
retrieval_log_fn = f'retr_{loc_name}.csv'
retr_log = pd.read_csv(retrieval_log_fn)

wind_multipliers = []

for r in retr_log.iterrows():
    granule_name = r[1]['GRANULE']
    obs_time = datetime.strptime(granule_name.split('_')[0], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)

    aglwind_info = get_geosfp_wind_agl(loc_data['LAT'], loc_data['LON'], obs_time, z_agl=500, cache=f'{CONFIG["geosfp"]}/')
    slvwind_info = get_geosfp_wind(loc_data['LAT'], loc_data['LON'], obs_time, cache=f'{CONFIG["geosfp"]}/')

    U = np.mean([slvwind_info['power_law'](k) for k in [200, 250, 350, 450, 550, 650, 750] ])
    # print(slvwind_info)
    ratio = aglwind_info['speed_ms']/U

    wind_multipliers.append(ratio)
    print(f"{granule_name}, {ratio}")
    # break

retr_log['WIND_CORR'] = wind_multipliers

KeyboardInterrupt: 

In [39]:
# ds_slv = xr.load_dataset('/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg1_2d_slv_Nx.20230824_0830.V01.nc4', engine='netcdf4')
# ds_slv.H1000

ds_3dv = xr.load_dataset('/orcd/data/dvaron/001/kgauld/GEOS_FP/GEOS.fp.asm.tavg3_3d_asm_Nv.20230824_0730.V01.nc4',
                        engine='netcdf4')
terrain_msl = float(
        ds_3dv.PHIS.sel(time='2023-08-24T08:30:00', lat=loc_data['LAT'], lon=loc_data['LON'], method="nearest").values
    )
terrain_msl/9.80665

638.284840914711

In [43]:
loc_data = LOCS['RIYADH_PLANT_9']
granule_name = '20230824T083907_2323606_023'
obs_time = datetime.strptime(granule_name.split('_')[0], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)

wind_info = get_geosfp_wind_agl(loc_data['LAT'], loc_data['LON'], obs_time, z_agl=500, cache=f'{CONFIG["geosfp"]}/')

/orcd/data/dvaron/001/kgauld/GEOS_FP//GEOS.fp.asm.tavg3_3d_asm_Nv.20230824_0730.V01.nc4


In [44]:
wind_info

{'speed_ms': 6.707522304544798,
 'dir_from_deg': 351.21780838030566,
 'u_ms': 1.024095603386131,
 'v_ms': -6.628882534868993,
 'z_used_m_agl': 500.0,
 'terrain_m_msl': 638.284840914711,
 'levels_z_agl_m': [69.55024209310147,
  207.07947793294522,
  343.836496975914,
  482.024485257164,
  621.743723538414,
  763.046946194664,
  906.005198147789,
  1050.561838772789,
  1196.874827054039,
  1344.982737210289,
  1494.836985257164,
  1646.773020413414,
  1826.630198147789,
  2062.192942288414,
  2329.296702054039,
  2602.770334866539,
  2883.333811429039,
  3170.660471585289,
  3541.811838772789,
  4002.968088772789,
  4485.146799710289,
  4992.745432522789,
  5533.309885647789,
  6109.457346585289,
  6723.163401272789,
  7381.086741116539,
  8093.670237210289,
  9012.779612210288,
  10104.076487210288,
  11180.113596585288,
  12233.996409085288,
  13264.781565335288,
  14264.278635647788,
  15229.968088772788,
  16171.625315335288,
  17097.390940335288,
  18027.510080960288,
  18995.426096

In [2]:
CONFIG

{'data_folder': '/orcd/data/dvaron/001/kgauld/EMIT/data',
 'results_folder': '/orcd/data/dvaron/001/kgauld/EMIT/results6_destriped',
 'plot_folder': '/orcd/data/dvaron/001/kgauld/EMIT/plots6_destriped_recolor',
 'geosfp': '/orcd/data/dvaron/001/kgauld/GEOS_FP',
 'hrrr': '/orcd/data/dvaron/001/kgauld/HRRR',
 'tropomi': '/orcd/data/dvaron/001/kgauld/TROPOMI',
 'campd_key': '/orcd/pool/005/dvaron_shared/kgauld/emit-nox-plumes/secrets/CAMPD_APIKEY',
 'retr_subdir': 'Retrievals',
 'tavg_subdir': 'Time_Average',
 'ps_subdir': 'Point_Source',
 'NOX_CSEC': '/orcd/home/002/kgauld/emit-nox-plumes/cross_sections/no2c_97.txt',
 'AMF_LUT': '/orcd/data/dvaron/001/kgauld/AMF/differential_amf.nc'}

In [3]:
retrieval_log_fn = f"{CONFIG['results_folder']}/{CONFIG['ps_subdir']}/source_retr_data.csv"
retr_log = pd.read_csv(retrieval_log_fn) 
retr_log

,LOC_NAME,GRANULE,CAMPD,SMOOTHED,BINNED,MEAN,AMF
0,New_Madrid_Power_Plant,20241201T171748_2433612_030,0.514833,0.723138,0.736963,0.359594,0.989857
1,New_Madrid_Power_Plant,20230823T170609_2323511_010,0.188026,0.151481,0.150878,0.101979,1.111175
2,New_Madrid_Power_Plant,20240815T191458_2422813_045,0.146677,0.206052,0.198356,0.111197,1.025909
3,New_Madrid_Power_Plant,20241012T201651_2428613_043,0.719227,1.036278,0.998043,0.605936,0.933956
4,New_Madrid_Power_Plant,20250401T172306_2509111_025,0.587039,0.478694,0.475381,0.289839,1.078142
...,...,...,...,...,...,...,...
65,Intermountain,20230927T214729_2327014_012,0.160136,0.465708,0.447016,0.321212,1.605108
66,Intermountain,20230202T193710_2303313_007,0.186295,0.053843,0.052649,0.024360,3.258090
67,Intermountain,20230620T193346_2317113_008,0.102811,0.350886,0.358586,0.243419,1.503576
68,Intermountain,20250424T164449_2511411_003,0.091421,0.102599,0.104581,0.054834,1.810623


In [4]:
smoothed_rates = {
}
for i in range(len(retr_log)):
    loc_name, granule_name = retr_log.iloc[i]['LOC_NAME'], retr_log.iloc[i]['GRANULE']
    smoothed = np.load(f"{CONFIG['results_folder']}/{CONFIG['ps_subdir']}/{loc_name}/{loc_name}_{granule_name}_smoothed.npy")
    smoothed_rates[granule_name] = [
        np.mean(smoothed[smoothed >= np.percentile(smoothed, k)]) for k in range(0,100,10)
    ]
    smoothed_rates[granule_name].append(len(smoothed)*0.06)

In [5]:
import pandas as pd

# Define column names
cols = [f"SMOOTHED_{k}" for k in range(0, 100, 10)] + ["PLUME_LENGTH"]

# Convert dictionary → DataFrame
smooth_df = pd.DataFrame.from_dict(smoothed_rates, orient="index", columns=cols)

# Merge into original dataframe using GRANULE
retr_log = retr_log.merge(smooth_df, left_on="GRANULE", right_index=True, how="left")

In [6]:
retr_log.to_csv(f"{CONFIG['results_folder']}/{CONFIG['ps_subdir']}/source_retr_data_wthr.csv")

In [9]:
retr_log

,LOC_NAME,GRANULE,CAMPD,SMOOTHED,BINNED,MEAN,AMF,SMOOTHED_0,SMOOTHED_10,SMOOTHED_20,SMOOTHED_30,SMOOTHED_40,SMOOTHED_50,SMOOTHED_60,SMOOTHED_70,SMOOTHED_80,SMOOTHED_90,PLUME_LENGTH
0,New_Madrid_Power_Plant,20241201T171748_2433612_030,0.514833,0.723138,0.736963,0.359594,0.989857,0.346599,0.371745,0.397568,0.421655,0.446340,0.475572,0.515490,0.571524,0.651224,0.723138,35.22
1,New_Madrid_Power_Plant,20230823T170609_2323511_010,0.188026,0.151481,0.150878,0.101979,1.111175,0.098693,0.107299,0.111704,0.114474,0.117185,0.120027,0.123632,0.128481,0.136023,0.151481,7.92
2,New_Madrid_Power_Plant,20240815T191458_2422813_045,0.146677,0.206052,0.198356,0.111197,1.025909,0.090301,0.097219,0.103658,0.111705,0.121071,0.132747,0.146979,0.166748,0.187346,0.206052,11.16
3,New_Madrid_Power_Plant,20241012T201651_2428613_043,0.719227,1.036278,0.998043,0.605936,0.933956,0.556672,0.610525,0.649163,0.690017,0.732305,0.777433,0.819143,0.866560,0.928839,1.036278,30.36
4,New_Madrid_Power_Plant,20250401T172306_2509111_025,0.587039,0.478694,0.475381,0.289839,1.078142,0.276843,0.300426,0.321532,0.343565,0.361683,0.379649,0.399063,0.420933,0.448956,0.478694,39.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Intermountain,20230927T214729_2327014_012,0.160136,0.465708,0.447016,0.321212,1.605108,0.319811,0.349621,0.369938,0.385596,0.398639,0.411428,0.423714,0.435653,0.450543,0.465708,6.48
66,Intermountain,20230202T193710_2303313_007,0.186295,0.053843,0.052649,0.024360,3.258090,0.021652,0.024077,0.026626,0.029460,0.032702,0.035998,0.040039,0.044633,0.048840,0.053843,41.70
67,Intermountain,20230620T193346_2317113_008,0.102811,0.350886,0.358586,0.243419,1.503576,0.235971,0.253937,0.269567,0.282198,0.294662,0.306553,0.316531,0.326366,0.337580,0.350886,12.90
68,Intermountain,20250424T164449_2511411_003,0.091421,0.102599,0.104581,0.054834,1.810623,0.053754,0.057267,0.060120,0.063018,0.066348,0.069964,0.074572,0.080873,0.089938,0.102599,19.02


In [2]:
retr_log = pd.read_csv(f"{CONFIG['results_folder']}/{CONFIG['ps_subdir']}/source_retr_data_wthr.csv")

In [3]:
retr_log

,Unnamed: 0,LOC_NAME,GRANULE,CAMPD,SMOOTHED,BINNED,MEAN,AMF,SMOOTHED_0,SMOOTHED_10,SMOOTHED_20,SMOOTHED_30,SMOOTHED_40,SMOOTHED_50,SMOOTHED_60,SMOOTHED_70,SMOOTHED_80,SMOOTHED_90,PLUME_LENGTH
0,0,New_Madrid_Power_Plant,20241201T171748_2433612_030,0.514833,0.723138,0.736963,0.359594,0.989857,0.346599,0.371745,0.397568,0.421655,0.446340,0.475572,0.515490,0.571524,0.651224,0.723138,35.22
1,1,New_Madrid_Power_Plant,20230823T170609_2323511_010,0.188026,0.151481,0.150878,0.101979,1.111175,0.098693,0.107299,0.111704,0.114474,0.117185,0.120027,0.123632,0.128481,0.136023,0.151481,7.92
2,2,New_Madrid_Power_Plant,20240815T191458_2422813_045,0.146677,0.206052,0.198356,0.111197,1.025909,0.090301,0.097219,0.103658,0.111705,0.121071,0.132747,0.146979,0.166748,0.187346,0.206052,11.16
3,3,New_Madrid_Power_Plant,20241012T201651_2428613_043,0.719227,1.036278,0.998043,0.605936,0.933956,0.556672,0.610525,0.649163,0.690017,0.732305,0.777433,0.819143,0.866560,0.928839,1.036278,30.36
4,4,New_Madrid_Power_Plant,20250401T172306_2509111_025,0.587039,0.478694,0.475381,0.289839,1.078142,0.276843,0.300426,0.321532,0.343565,0.361683,0.379649,0.399063,0.420933,0.448956,0.478694,39.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,65,Intermountain,20230927T214729_2327014_012,0.160136,0.465708,0.447016,0.321212,1.605108,0.319811,0.349621,0.369938,0.385596,0.398639,0.411428,0.423714,0.435653,0.450543,0.465708,6.48
66,66,Intermountain,20230202T193710_2303313_007,0.186295,0.053843,0.052649,0.024360,3.258090,0.021652,0.024077,0.026626,0.029460,0.032702,0.035998,0.040039,0.044633,0.048840,0.053843,41.70
67,67,Intermountain,20230620T193346_2317113_008,0.102811,0.350886,0.358586,0.243419,1.503576,0.235971,0.253937,0.269567,0.282198,0.294662,0.306553,0.316531,0.326366,0.337580,0.350886,12.90
68,68,Intermountain,20250424T164449_2511411_003,0.091421,0.102599,0.104581,0.054834,1.810623,0.053754,0.057267,0.060120,0.063018,0.066348,0.069964,0.074572,0.080873,0.089938,0.102599,19.02


In [6]:
L = retr_log["PLUME_LENGTH"].to_numpy(dtype=float)

# 90 for short plumes, 60 for long plumes
x_thresh = np.interp(L, [5, 25], [90, 60])
x_thresh = 10 * np.round(x_thresh / 10).astype(int)
x_thresh = np.clip(x_thresh, 0, 90)
retr_log['x_thresh'] = x_thresh

In [19]:
csv_savefn = f"{CONFIG['data_folder']}/metadata_FULL.csv"
data = pd.read_csv(csv_savefn)
factors = []

for r in retr_log.iterrows():
    r = r[1]
    loc_info = REFERENCE_PLANTS[r['LOC_NAME']]
    granule_data = data[data['GRANULE'] == r['GRANULE']]
    mean_windspeed = granule_data[["GEOSFP_50M_SPD", "HRRR_10M_SPD", "HRRR_AGL_SPD"]].iloc[0].mean()
    use_windspeed = granule_data["HRRR_AGL_SPD"].iloc[0]
    factors.append(use_windspeed/mean_windspeed)


retr_log['HRRR_SCALE'] = factors

In [20]:
retr_log.to_csv(f"{CONFIG['results_folder']}/{CONFIG['ps_subdir']}/source_retr_data_whrscale.csv")

In [21]:
csv_savefn = f"{CONFIG['data_folder']}/metadata_FULL.csv"
data = pd.read_csv(csv_savefn)
factors = []

for r in retr_log.iterrows():
    r = r[1]
    loc_data = REFERENCE_PLANTS[r['LOC_NAME']]
    obs_time = datetime.strptime(r['GRANULE'].split('_')[0], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
    granule_data = data[data['GRANULE'] == r['GRANULE']]
    mean_windspeed = granule_data[["GEOSFP_50M_SPD", "HRRR_10M_SPD", "HRRR_AGL_SPD"]].iloc[0].mean()

    wind_info = get_geosfp_wind(loc_data['LAT'], loc_data['LON'], obs_time, cache=f'{CONFIG["geosfp"]}/')
    Z = [250, 350, 450, 550, 650, 750]          # m
    U = [wind_info["power_law"](z) for z in Z]  # or wind["log_law"](z)
    use_windspeed = np.mean(U)
    
    factors.append(use_windspeed/mean_windspeed)


retr_log['GEOSFP_SCALE'] = factors

In [22]:
retr_log.to_csv(f"{CONFIG['results_folder']}/{CONFIG['ps_subdir']}/source_retr_data_whr_geosscale.csv")

In [10]:
from get_hrrr import get_hrrr_wind_10m, get_hrrr_wind_agl

loc_name = 'XAI_CENTER'

datestamp = [k.split('.')[0].split('_')[-3] for k in glob.glob(f"{CONFIG['data_folder']}/{loc_name}/*RAD*")]

ltlat, ltlon = LOCS[loc_name]['LAT'], LOCS[loc_name]['LON']

for d in datestamp:
    obs_time = datetime.strptime(d, "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
    hrrr_agl_info = get_hrrr_wind_agl(ltlat, ltlon, obs_time, layer=(300,700), cache=f'{CONFIG["hrrr"]}')
    print(f"{d}\t{hrrr_agl_info['speed_ms']}")

2026-04-24 17:11:14,642 INFO Downloading https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20240420/conus/hrrr.t17z.wrfprsf00.grib2
2026-04-24 17:11:25,570 INFO hrrr.t18z.wrfprsf00.grib2 already downloaded


20240420T175158	10.820908791671478


2026-04-24 17:11:42,172 INFO hrrr.t15z.wrfprsf00.grib2 already downloaded


20240330T183753	9.316357770765048


2026-04-24 17:11:59,096 INFO Downloading https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20250811/conus/hrrr.t20z.wrfprsf00.grib2


...

20240801T173126	7.968088676338532


2026-04-24 17:12:58,551 INFO Downloading https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20231223/conus/hrrr.t16z.wrfprsf00.grib2


20250415T191149	7.076986660180205


2026-04-24 17:13:11,534 INFO Downloading https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20240622/conus/hrrr.t16z.wrfprsf00.grib2


20231223T165831	6.808004593229594
20240622T165422	4.354505041032893


In [5]:
CONFIG

{'data_folder': '/orcd/data/dvaron/001/kgauld/EMIT/data',
 'results_folder': '/orcd/data/dvaron/001/kgauld/EMIT/results6',
 'plot_folder': '/orcd/data/dvaron/001/kgauld/EMIT/plots6_destriped_recolor',
 'geosfp': '/orcd/data/dvaron/001/kgauld/GEOS_FP',
 'hrrr': '/orcd/data/dvaron/001/kgauld/HRRR',
 'tropomi': '/orcd/data/dvaron/001/kgauld/TROPOMI',
 'campd_key': '/orcd/pool/005/dvaron_shared/kgauld/emit-nox-plumes/secrets/CAMPD_APIKEY',
 'retr_subdir': 'Retrievals',
 'tavg_subdir': 'Time_Average',
 'ps_subdir': 'Point_Source',
 'NOX_CSEC': '/orcd/home/002/kgauld/emit-nox-plumes/cross_sections/no2c_97.txt',
 'AMF_LUT': '/orcd/data/dvaron/001/kgauld/AMF/differential_amf.nc'}